# Pipeline do projeto

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# bibliotecas gerais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# classes do projeto
from src.feature_engineering import EngenhariaFeaturesTelco
from src.baseline_training import BaselineTrainer
from src.otimizacao import OtimizadorMLP
from src.model_selection import ModelComparator


c:\Users\gui08\OneDrive\Documentos\Tech_challenge_1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Aquisição dos dados

In [2]:
caminho_dados = PROJECT_ROOT / 'data' / 'Telco_customer_churn.xlsx'
coluna_alvo_binaria = "target"


df = pd.read_excel(caminho_dados)


## Engenharia de Features

In [3]:
feature_engineering = EngenhariaFeaturesTelco()
df_modelo = feature_engineering.processar(
    caminho_entrada=caminho_dados,
    caminho_saida=PROJECT_ROOT / 'data' / 'processed' / 'telco_feature_engineered.csv'
)

## Treinamento Dummy Baseline

In [4]:
baseline = BaselineTrainer(
    input_path=PROJECT_ROOT / 'data' / 'processed' / 'telco_feature_engineered.csv',
    target='Churn Value',
    test_size=0.2,
    random_seed=42,
)
baseline_payload = baseline.executar()

In [5]:
results = baseline_payload['results']  # dict com keys 'dummy_classifier' e 'logistic_regression'
metrics_df = pd.DataFrame(results).T
display(metrics_df)

,accuracy,precision,recall,f1,roc_auc
dummy_classifier,0.734564,0.000000,0.000000,0.000000,0.500000
logistic_regression,0.745209,0.513228,0.778075,0.618491,0.847932


## Otimização do MLP

In [6]:
otimizador = OtimizadorMLP(
    input_path=PROJECT_ROOT / 'data' / 'processed' / 'telco_feature_engineered.csv',
    n_trials=30,
    random_seed=42,
)
study, melhor_trial = otimizador.rodar()
melhor_trial

[I 2026-09-01 16:10:17,094] A new study created in memory with name: mlp_classifier_optimization
[I 2026-09-01 16:10:28,948] Trial 0 finished with value: 0.6156716417910447 and parameters: {'hidden_layer_sizes': '128,64', 'max_iter': 200, 'alpha': 4.207053950287936e-06, 'learning_rate_init': 0.00013066739238053285, 'batch_size': 32}. Best is trial 0 with value: 0.6156716417910447.
[I 2026-09-01 16:10:36,855] Trial 1 finished with value: 0.6115384615384616 and parameters: {'hidden_layer_sizes': '128,64', 'max_iter': 200, 'alpha': 5.415244119402541e-06, 'learning_rate_init': 0.0004059611610484307, 'batch_size': 32}. Best is trial 0 with value: 0.6156716417910447.
[I 2026-09-01 16:10:40,402] Trial 2 finished with value: 0.6483516483516484 and parameters: {'hidden_layer_sizes': '64', 'max_iter': 400, 'alpha': 0.0013826232179369874, 'learning_rate_init': 0.00025081156860452336, 'batch_size': 64}. Best is trial 2 with value: 0.6483516483516484.
[I 2026-09-01 16:10:51,605] Trial 3 finished wi

{'params': {'hidden_layer_sizes': '64',
  'max_iter': 600,
  'alpha': 2.7786154527585736e-06,
  'learning_rate_init': 0.006158482954754892,
  'batch_size': 128},
 'f1_validation': 0.6583184257602862,
 'number': 25}

In [7]:
# O comparador treina Logistic Regression, Random Forest e MLPClassifier.
comparador = ModelComparator(
    input_path=PROJECT_ROOT / 'data' / 'processed' / 'telco_feature_engineered.csv',
    test_size=0.2,
    validation_size=0.2,
    cv_folds=5,
    random_seed=42,
    mlp_parameters=melhor_trial['params'],
)
resultado_comparacao = comparador.executar()
comparacao = resultado_comparacao['comparison']
comparacao

,model,cv_f1_mean,cv_f1_std,cv_recall_mean,cv_recall_std,cv_roc_auc_mean,cv_roc_auc_std
0,logistic_regression,0.641268,0.012361,0.815234,0.017640,0.857671,0.007307
1,random_forest,0.620090,0.015504,0.673124,0.033175,0.843002,0.009473
2,mlp_classifier,0.589151,0.051761,0.556900,0.084334,0.847609,0.011713


In [8]:
comparacao.to_csv(PROJECT_ROOT / 'results' / 'model_comparison.csv', index=False)
comparacao

,model,cv_f1_mean,cv_f1_std,cv_recall_mean,cv_recall_std,cv_roc_auc_mean,cv_roc_auc_std
0,logistic_regression,0.641268,0.012361,0.815234,0.017640,0.857671,0.007307
1,random_forest,0.620090,0.015504,0.673124,0.033175,0.843002,0.009473
2,mlp_classifier,0.589151,0.051761,0.556900,0.084334,0.847609,0.011713


## Métricas

In [9]:
comparacao.set_index('model')[['cv_f1_mean', 'cv_f1_std', 'cv_recall_mean', 'cv_roc_auc_mean']]

,cv_f1_mean,cv_f1_std,cv_recall_mean,cv_roc_auc_mean
model,,,,
logistic_regression,0.641268,0.012361,0.815234,0.857671
random_forest,0.620090,0.015504,0.673124,0.843002
mlp_classifier,0.589151,0.051761,0.556900,0.847609


In [13]:
print(f"Modelo campeão por F1 de validação cruzada: {resultado_comparacao['champion_name']}")
print(resultado_comparacao['test_metrics'])
#print(f"Artefato: {resultado_comparacao['model_path']}")

Modelo campeão por F1 de validação cruzada: logistic_regression
{'accuracy': 0.7437899219304471, 'precision': 0.5114235500878734, 'recall': 0.7780748663101604, 'f1': 0.6171792152704135, 'roc_auc': 0.8477253351933659}


## Artefatos do modelo campeão